World Cup XGBoost Application
    
    Datasets:
          World Cup Players Statisics (2023 - 2025) 
          Qualified 2026 World Cup Teams International Match Results (2018 - 2025)
          World Cup FIFA Elo Rating (2018 - 2026)


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson as poisson_dist
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv(r'C:\Users\ngman\OneDrive\UNC MADS\DATA780 Machine Learning\Group Project - FIFA\780_Group_Project_2026\Player_Stats_XGBoost\raw_pull\all_players_raw_stats.csv')
df.head()

,player_id,name,age,nationality,season,club,competition,country,appearances,minutes,...,assists,passes_total,passes_key,pass_accuracy_pct,shots_total,shots_on_target,tackles_total,duels_total,duels_won,roster_nation
0,730,T. Courtois,33,Belgium,2023,Real Madrid,La Liga,Spain,5.0,333.0,...,0.0,98.0,1.0,NaN,NaN,NaN,1.0,3.0,3.0,Belgium
1,730,T. Courtois,33,Belgium,2023,Real Madrid,Copa del Rey,Spain,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Belgium
2,730,T. Courtois,33,Belgium,2023,Real Madrid,UEFA Champions League,World,1.0,90.0,...,0.0,18.0,NaN,NaN,NaN,NaN,NaN,2.0,1.0,Belgium
3,730,T. Courtois,33,Belgium,2023,Real Madrid,Friendlies Clubs,World,2.0,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Belgium
4,730,T. Courtois,33,Belgium,2023,Belgium,Euro Championship - Qualification,World,2.0,180.0,...,NaN,56.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Belgium


In [4]:
#Player Data Cleaning for XGBoost Model
df = df[df["minutes"] > 0].copy()
df = df.drop(columns=["pass_accuracy_pct"])

international_keywords = [
    "World Cup", "Qualification", "Nations League", "Cup of Nations",
    "Euro Championship", "Copa America", "Friendlies", "Gold Cup", "Asian Cup",
]

df["is_international"] = df["competition"].str.contains(
    "|".join(international_keywords), na=False
)

gk_mask = df["position"] == "Goalkeeper"
df.loc[gk_mask, ["shots_total", "shots_on_target"]] = np.nan

In [5]:
#2. Player Data Aggregation for XGBoost Model
sum_cols = [
    "appearances", "minutes", "goals", "assists", "passes_total",
    "passes_key", "shots_total", "shots_on_target", "tackles_total",
    "duels_total", "duels_won",
]

agg = df.groupby(
    ["player_id", "name", "roster_nation", "is_international"], as_index=False
)[sum_cols].sum(min_count=1)

#Player performance by rating and minutes played to create minutes weighted average rating for each player
df["rating_x_min"] = df["rating"] * df["minutes"]
rating_agg = df.groupby(["player_id", "is_international"], as_index=False).agg(
    rating_x_min=("rating_x_min", "sum"), minutes_for_rating=("minutes", "sum")
)
rating_agg["avg_rating"] = rating_agg["rating_x_min"] / rating_agg["minutes_for_rating"]
agg = agg.merge(
    rating_agg[["player_id", "is_international", "avg_rating"]],
    on=["player_id", "is_international"], how="left"
)

club = agg[~agg["is_international"]].drop(columns=["is_international"]).add_prefix("club_")
club = club.rename(columns={
    "club_player_id": "player_id", "club_name": "name", "club_roster_nation": "roster_nation"
})
intl = agg[agg["is_international"]].drop(columns=["is_international"]).add_prefix("intl_")
intl = intl.rename(columns={"intl_player_id": "player_id"})
 
player_features = club.merge(
    intl.drop(columns=[c for c in intl.columns if c.endswith("name") or c.endswith("roster_nation")]),
    on="player_id", how="left"
)

In [6]:
#3. Team-Level Features Aggregation for XGBoost Model - averages every player's stats across the full 26-man roster to get one row per nation (48 total).
feature_cols = [
    c for c in player_features.columns
    if (c.startswith("club_") or c.startswith("intl_")) and c != "club_name"
]
team_features = player_features.groupby("roster_nation")[feature_cols].mean().reset_index()
team_features.to_csv("team_features_final.csv", index=False)

In [7]:
#4. ELO Ratings (Drop Exit Round Column and Rename Country Column to Avoid Collisions). 
elo = pd.read_csv("elo_ratings_wc2026.csv")

elo = elo.drop(columns=["wc2026_exit_round"]) #this provides model with no information about the outcome of the tournament which would bias the model heavily.
 
elo["snapshot_date"] = pd.to_datetime(elo["snapshot_date"])
# Renamed to avoid colliding with the match data's own "country" column
# (host country of the match, a different thing entirely).
elo = elo.rename(columns={"country": "elo_country_lookup"})

In [8]:
#5. Merge ELO Ratings with Match Results
matches = pd.read_csv("filtered_results_2018_2026.csv")
matches["date"] = pd.to_datetime(matches["date"])

NAME_MAP_TO_ELO = {"Czech Republic": "Czechia"}
matches["home_team_elo"] = matches["home_team"].replace(NAME_MAP_TO_ELO)
matches["away_team_elo"] = matches["away_team"].replace(NAME_MAP_TO_ELO)

elo_cols = [
    "rank", "rating", "rank_max", "rating_max", "rank_avg", "rating_avg",
    "matches_total", "wins", "losses", "draws", "goals_for", "goals_against",
]
elo_sorted = elo.sort_values("snapshot_date")
matches_sorted = matches.sort_values("date")


home_rename = {c: f"home_elo_{c}" for c in elo_cols}
step1 = pd.merge_asof(
    matches_sorted, elo_sorted[["snapshot_date", "elo_country_lookup"] + elo_cols],
    left_on="date", right_on="snapshot_date",
    left_by="home_team_elo", right_by="elo_country_lookup", direction="backward",
)
step1 = step1.rename(columns=home_rename).drop(columns=["snapshot_date", "elo_country_lookup"])
 
away_rename = {c: f"away_elo_{c}" for c in elo_cols}
step2 = pd.merge_asof(
    step1.sort_values("date"), elo_sorted[["snapshot_date", "elo_country_lookup"] + elo_cols],
    left_on="date", right_on="snapshot_date",
    left_by="away_team_elo", right_by="elo_country_lookup", direction="backward",
)
step2 = step2.rename(columns=away_rename).drop(columns=["snapshot_date", "elo_country_lookup"])

In [9]:
#6. Player/Team Feature Mapping

home_pf = team_features.add_prefix("home_pf_")
away_pf = team_features.add_prefix("away_pf_")
 
full = step2.merge(home_pf, left_on="home_team", right_on="home_pf_roster_nation", how="left")
full = full.merge(away_pf, left_on="away_team", right_on="away_pf_roster_nation", how="left")
 
full.shape

(3832, 88)

In [10]:
#7. Difference Features
player_cols = [c.replace("home_pf_", "") for c in full.columns
               if c.startswith("home_pf_") and c != "home_pf_roster_nation"]
 
for c in elo_cols:
    full[f"diff_elo_{c}"] = full[f"home_elo_{c}"] - full[f"away_elo_{c}"]
for c in player_cols:
    full[f"diff_pf_{c}"] = full[f"home_pf_{c}"] - full[f"away_pf_{c}"]
 
diff_cols = [f"diff_elo_{c}" for c in elo_cols] + [f"diff_pf_{c}" for c in player_cols]
full.to_csv("final_match_feature_table.csv", index=False)

In [11]:
#8. Train/Test Split
train = full[full["date"] < "2026-06-11"].copy() #before tournament starts
test = full[(full["tournament"] == "FIFA World Cup") & (full["date"] >= "2026-06-11")].copy()
 
for d in (train, test):
    d["result"] = np.select(
        [d["home_score"] > d["away_score"], d["home_score"] == d["away_score"]],
        [0, 1], default=2
    )  # 0 = home win, 1 = draw, 2 = away win
 
print(f"Train matches: {len(train)}, Test matches (real WC games): {len(test)}")


Train matches: 3728, Test matches (real WC games): 104


In [12]:
#9. Poisson Baseline
home_long = train.rename(columns={
    "home_team": "team", "away_team": "opponent",
    "home_score": "goals", "away_score": "opponent_goals"
}).copy()
home_long["is_home"] = 1
away_long = train.rename(columns={
    "away_team": "team", "home_team": "opponent",
    "away_score": "goals", "home_score": "opponent_goals"
}).copy()
away_long["is_home"] = 0
train_long = pd.concat([home_long, away_long], ignore_index=True)
 
poisson_model = smf.glm(
    formula="goals ~ team + opponent + is_home", data=train_long, family=sm.families.Poisson()
).fit()

In [17]:
#10. Match Predicion with Poisson Model
poisson_predictions = []
for _, row in test.iterrows():
    lam_home = poisson_model.predict(pd.DataFrame({
        "team": [row["home_team"]], "opponent": [row["away_team"]], "is_home": [1]
    })).iloc[0]
    lam_away = poisson_model.predict(pd.DataFrame({
        "team": [row["away_team"]], "opponent": [row["home_team"]], "is_home": [0]
    })).iloc[0]
 
    home_probs = poisson_dist.pmf(np.arange(0, 11), lam_home)
    away_probs = poisson_dist.pmf(np.arange(0, 11), lam_away)
    score_matrix = np.outer(home_probs, away_probs)
 
    p_home_win = np.tril(score_matrix, -1).sum()
    p_draw = np.trace(score_matrix)
    p_away_win = np.triu(score_matrix, 1).sum()
 
    predicted = np.argmax([p_home_win, p_draw, p_away_win])
    poisson_predictions.append(predicted)
 
test["poisson_pred"] = poisson_predictions
poisson_accuracy = accuracy_score(test["result"], test["poisson_pred"])
print(f"POISSON accuracy on all {len(test)} real WC matches: {poisson_accuracy:.1%}")

POISSON accuracy on all 104 real WC matches: 67.3%


In [ ]:
#11. XGBoost Model Training
train_xgb = train.dropna(subset=diff_cols)
test_xgb = test.dropna(subset=diff_cols)
print(f"XGBoost usable matches: {len(test_xgb)} of {len(test)} "
      f"({len(test) - len(test_xgb)} dropped for missing data)")
 
xgb_model = XGBClassifier(
    objective="multi:softprob", num_class=3, eval_metric="mlogloss",
    n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42, reg_lambda=2.0,
)#lambda regularization to reduce overfitting. This makes the tree more reluctant to create new branches.
xgb_model.fit(train_xgb[diff_cols], train_xgb["result"])
 
xgb_predictions = xgb_model.predict(test_xgb[diff_cols])
xgb_accuracy = accuracy_score(test_xgb["result"], xgb_predictions)
print(f"XGBOOST accuracy ({len(test_xgb)} matches): {xgb_accuracy:.1%}")
print()
print(classification_report(test_xgb["result"], xgb_predictions,
                             target_names=["home_win", "draw", "away_win"]))

XGBoost usable matches: 83 of 104 (21 dropped for missing data)
XGBOOST accuracy (83 matches): 63.9%

              precision    recall  f1-score   support

    home_win       0.68      0.86      0.76        37
        draw       0.25      0.22      0.24        18
    away_win       0.85      0.61      0.71        28

    accuracy                           0.64        83
   macro avg       0.59      0.56      0.57        83
weighted avg       0.64      0.64      0.63        83



In [15]:
poisson_same_subset_accuracy = accuracy_score(test_xgb["result"], test_xgb["poisson_pred"])
 
print("=== FAIR COMPARISON (both models scored on the identical matches) ===")
print(f"Poisson: {poisson_same_subset_accuracy:.1%}")
print(f"XGBoost: {xgb_accuracy:.1%}")
 
# Which features did XGBoost actually rely on?
importances = pd.Series(xgb_model.feature_importances_, index=diff_cols).sort_values(ascending=False)
print("\nTop 10 most important features:")
print(importances.head(10))

=== FAIR COMPARISON (both models scored on the identical matches) ===
Poisson: 68.7%
XGBoost: 63.9%

Top 10 most important features:
diff_pf_club_shots_on_target    0.133368
diff_elo_rating                 0.059119
diff_pf_club_passes_key         0.040739
diff_pf_club_minutes            0.033026
diff_pf_club_avg_rating         0.032592
diff_pf_club_assists            0.032067
diff_pf_club_appearances        0.031370
diff_elo_rank                   0.030563
diff_pf_intl_duels_total        0.028103
diff_pf_club_duels_won          0.027677
dtype: float32


In [16]:
print("=" * 60)
print("POISSON classification report (same matches as XGBoost):")
print("=" * 60)
print(classification_report(
    test_xgb["result"], test_xgb["poisson_pred"],
    target_names=["home_win", "draw", "away_win"]
))
 
print("=" * 60)
print("XGBOOST classification report:")
print("=" * 60)
print(classification_report(
    test_xgb["result"], xgb_predictions,
    target_names=["home_win", "draw", "away_win"]
))
 
# Pull out JUST the draw row from each report for a direct side-by-side
from sklearn.metrics import precision_recall_fscore_support
 
poisson_prf = precision_recall_fscore_support(
    test_xgb["result"], test_xgb["poisson_pred"], labels=[0, 1, 2]
)
xgb_prf = precision_recall_fscore_support(
    test_xgb["result"], xgb_predictions, labels=[0, 1, 2]
)
 
draw_comparison = pd.DataFrame({
    "poisson_precision": [poisson_prf[0][1]],
    "poisson_recall":    [poisson_prf[1][1]],
    "poisson_f1":        [poisson_prf[2][1]],
    "xgboost_precision": [xgb_prf[0][1]],
    "xgboost_recall":    [xgb_prf[1][1]],
    "xgboost_f1":        [xgb_prf[2][1]],
}, index=["draw"])
 
print("=" * 60)
print("DRAW-ONLY comparison:")
print("=" * 60)
print(draw_comparison.T)
 
# How many actual draws were there, and how many did each model catch?
actual_draws = (test_xgb["result"] == 1).sum()
poisson_draws_predicted = (test_xgb["poisson_pred"] == 1).sum()
poisson_draws_correct = ((test_xgb["result"] == 1) & (test_xgb["poisson_pred"] == 1)).sum()
xgb_draws_predicted = (xgb_predictions == 1).sum()
xgb_draws_correct = ((test_xgb["result"] == 1) & (xgb_predictions == 1)).sum()
 
print(f"\nActual draws in test set: {actual_draws}")
print(f"Poisson predicted 'draw' {poisson_draws_predicted} times, correct {poisson_draws_correct}/{actual_draws}")
print(f"XGBoost predicted 'draw' {xgb_draws_predicted} times, correct {xgb_draws_correct}/{actual_draws}")

POISSON classification report (same matches as XGBoost):
              precision    recall  f1-score   support

    home_win       0.64      0.92      0.76        37
        draw       0.00      0.00      0.00        18
    away_win       0.77      0.82      0.79        28

    accuracy                           0.69        83
   macro avg       0.47      0.58      0.52        83
weighted avg       0.54      0.69      0.60        83

XGBOOST classification report:
              precision    recall  f1-score   support

    home_win       0.68      0.86      0.76        37
        draw       0.25      0.22      0.24        18
    away_win       0.85      0.61      0.71        28

    accuracy                           0.64        83
   macro avg       0.59      0.56      0.57        83
weighted avg       0.64      0.64      0.63        83

DRAW-ONLY comparison:
                       draw
poisson_precision  0.000000
poisson_recall     0.000000
poisson_f1         0.000000
xgboost_precisio

c:\Users\ngman\.julia\conda\3\x86_64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ngman\.julia\conda\3\x86_64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ngman\.julia\conda\3\x86_64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh